# AegisPlan: Fast QLoRA Fine-Tuning with Unsloth
**Autonomous Neuro-Symbolic Healthcare Spatial Synthesis Engine**

This notebook fine-tunes an LLM to generate code-compliant `FML-DSL` clinical floorplans using **Unsloth** (2x faster, 70% less VRAM).
Runs on **Google Colab Free Tier (T4 16GB GPU)** or **Colab Pro (L4 / A100)**.

### Step 1: Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install --no-deps unsloth "xformers<0.0.29" trl peft accelerate bitsandbytes datasets

### Step 2: Load Model with Unsloth 4-bit Quantization
Choose your base model. Both `Qwen2.5-7B` and `Llama-3.1-8B` have native 1-click Unsloth kernels on Colab T4.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # None for auto detection (Float16 on T4, Bfloat16 on Ampere+)
load_in_4bit = True

# Select base model:
# 1. "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" (Fastest & best structured code generation on T4)
# 2. "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
# 3. "nvidia/NVIDIA-Nemotron-Nano-9B-v2" (Requires A100 / L4 on Colab Pro)
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Configure LoRA (r=32, alpha=64 from Master Plan Section 5)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0, # Optimized = 0
    bias="none",
    use_gradient_checkpointing="unsloth", # 30% less VRAM
    random_state=42,
)

### Step 3: Upload & Load the 100% Certified AegisPlan Dataset
Upload `dataset/sft_clinical_curriculum.jsonl` using the Colab file browser (left panel), or fetch directly.

In [ ]:
from datasets import load_dataset
import os

dataset_file = "sft_clinical_curriculum.jsonl"
if not os.path.exists(dataset_file):
    print(f"Please upload {dataset_file} via the Colab files tab!")
else:
    dataset = load_dataset("json", data_files=dataset_file, split="train")
    print(f"Successfully loaded {len(dataset)} verified clinical training samples!")

    def formatting_prompts_func(examples):
        texts = []
        for msgs in examples["messages"]:
            text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            texts.append(text)
        return {"text": texts}

    formatted_dataset = dataset.map(formatting_prompts_func, batched=True)
    print("Dataset formatted with chat template. Example preview:")
    print(formatted_dataset[0]["text"][:350] + "...")

### Step 4: Run Fine-Tuning with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=150,  # ~3 epochs on 500 samples
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="aegisplan_checkpoints",
        report_to="none",
    ),
)

# Start training
trainer_stats = trainer.train()
print("Training Complete!")

### Step 5: Test Generation (Inference)

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = "Design an outpatient clinic with waiting room, central corridor, 3 exam rooms, and ADA restroom."

messages = [
    {"role": "system", "content": "You are AegisPlan, an autonomous AI architect specializing in U.S. Healthcare Outpatient Facilities.\nOutput ONLY valid FML-DSL code matching the exact grammar: BOUNDARY RECT 0 0 <W> <H>, ROOM, DOOR, FDOOR, END_PLAN."},
    {"role": "user", "content": f"Generate an outpatient clinic floorplan for boundary RECT 0 0 16000 12000.\nClinical Brief: {test_prompt}\nEmit strictly valid FML-DSL:"}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.7, top_p=0.9)
generated_fml = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("=== GENERATED FML-DSL OUTPUT ===")
print(generated_fml)

### Step 6: Save Model & Export (LoRA / Hugging Face Hub / GGUF)

In [ ]:
# 1. Save LoRA Adapter Locally
model.save_pretrained("aegisplan_lora_adapter")
tokenizer.save_pretrained("aegisplan_lora_adapter")
print("Saved LoRA adapter to ./aegisplan_lora_adapter")

# 2. Optional: Push to Hugging Face Hub
# from huggingface_hub import login
# login()  # enter HF write token
# model.push_to_hub_merged("your-username/aegisplan-clinical-7b", tokenizer, save_method="merged_16bit")

# 3. Optional: Export directly to GGUF (q4_k_m for llama.cpp / vLLM)
# model.save_pretrained_gguf("aegisplan_gguf", tokenizer, quantization_method="q4_k_m")